# Not as Important Fire Data Information 

In [1]:
from pathlib import Path
import pandas as pd

# ---------- Files ----------
CSV_PATH = Path("Copy of all data") / "fire_data_2010.csv"
OUTPUT_DIR = Path("wildfire_2010_outputs")   # plots (PNG) and the HTML map are saved here
OUTPUT_DIR.mkdir(exist_ok=True)

# Load the raw data as a dataFrame. parse_dates turns acq_date into proper dates.
raw = pd.read_csv(CSV_PATH, parse_dates=["acq_date"])

### What are the differences between the two satellites?

**Terra** and **Aqua** carry identical MODIS instruments but cross California at different times of day (Terra at about 10:30 am/pm local time, Aqua at about 1:30 pm/am), so they see fires at different stages of the day.

In [2]:
raw.groupby("satellite").agg(
    detections=("frp", "size"),
    mean_frp=("frp", "mean"),
    median_frp=("frp", "median"),
    mean_brightness=("brightness", "mean"),
    mean_confidence=("confidence", "mean"),
).round(1)

,detections,mean_frp,median_frp,mean_brightness,mean_confidence
satellite,,,,,
Aqua,1532,73.6,23.6,328.7,69.7
Terra,1419,44.1,18.0,320.9,65.6


Aqua records slightly more detections (1,532 vs 1,419) and more intense ones (mean FRP ~74 vs ~44 MW).

Note: Terra Observations to Cease on December 1, 2026.
How will this affect the data? 
Can only train the models on data before this (not too much of an issue, given the timeframe of the project, unlikley to be a wildfire between now and April).

https://www.earthdata.nasa.gov/data/alerts-outages/new-terra-observations-cease-december-1-2026

### What does `brightness` mean, and does it depend on day/night?

`brightness` is a **temperature in Kelvin**, not a visual brightness. It is the pixel's temperature in the fire-sensitive mid-infrared channel, so higher values mean a hotter fire (or hotter ground).

In [3]:
raw.groupby("daynight").agg(
    detections=("frp", "size"),
    mean_brightness_K=("brightness", "mean"),
    mean_bright_t31_K=("bright_t31", "mean"),
    mean_frp=("frp", "mean"),
).round(1)

,detections,mean_brightness_K,mean_bright_t31_K,mean_frp
daynight,,,,
D,2433,327.0,299.2,61.5
N,518,315.5,289.1,49.7


It depends on day/night. Day-time brightness is higher (~327 K vs ~316 K) and so is the background temperature `bright_t31` (~299 K vs ~289 K). Also, ~82 % of detections are daytime.

Channel 31 is a better background measure. 

Fire emission peaks in the mid-IR. By Wien's displacement law (λₘₐₓ = b/T, where b is a constant), a surface at 300 K (normal ground) peaks around 10 µm, while a fire at 800 to 1000 K peaks around 3 to 4 µm. Channel 21/22 (about 3.96 µm) sits near the fire peak.

Channel 31 (about 11 µm) is much less sensitive to fire. It sits near the peak of ambient Earth emission. The pixel's 11 µm brightness temperature is dominated by the cooler background (soil, vegetation, etc.), therefore it approximates the temperature of the surrounding non-burning surface.

The difference between the two (T21 minus T31) is a key fire signal. A large gap means something much hotter than the background is present.